# USAERO – EDA bản gọn: nhanh, chạy được nhiều năm, dễ học

**Ba ý tưởng chính (đọc xong là hiểu cả notebook):**
1. **Tính một lần, cộng lại nhiều lần.** Mỗi tháng chỉ đọc dữ liệu thô một lần, lưu các cột *cộng được* (đếm, tổng). Ngày, tháng, năm, mùa vụ chỉ là phép cộng các bảng nhỏ này; tỷ lệ luôn tính lại từ tổng.
2. **Box-plot chính xác từ biểu đồ tần suất.** Phân vị không cộng được, nhưng "số chuyến có delay = v phút" thì cộng được. Nhờ đó box-plot mọi mức không cần quét lại dữ liệu thô.
3. **Null có ngữ nghĩa.** `ARR_DELAY`/`DEP_DELAY` null = hủy/chuyển hướng (không điền 0). Cột nguyên nhân trễ null = không trễ ≥15' (điền 0).

Chạy lần lượt từ trên xuống. Build có thể ngắt và chạy lại: tháng đã xong sẽ được bỏ qua.

### 1. Môi trường, cấu hình, quy ước chỉ số

In [1]:
import os, sys, glob, re, time, json, io, contextlib, datetime as dt
if os.path.isdir("/opt/spark"):
    os.environ.setdefault("SPARK_HOME", "/opt/spark")
try:
    import pyspark
except ModuleNotFoundError:
    sys.path.insert(0, "/opt/spark/python")
    sys.path.extend(glob.glob("/opt/spark/python/lib/py4j-*-src.zip"))
    import pyspark
from pyspark.sql import SparkSession, Window, functions as F, types as T
from pyspark import StorageLevel

# ── Đường dẫn ──
PROCESSED    = os.environ.get("AERO_PROCESSED", "/aero/processed")       # đầu ra của 06_batch_process_all.py
ANALYTICS    = os.environ.get("AERO_ANALYTICS", "/aero/analytics/eda")   # nơi ghi kết quả phân tích
HDFS_URI     = os.environ.get("AERO_HDFS", "hdfs://namenode:9000")
WAREHOUSE    = os.environ.get("AERO_WAREHOUSE", (HDFS_URI + "/aero/warehouse") if HDFS_URI else "/tmp/aero_warehouse")
EVIDENCE_DIR = os.environ.get("AERO_EVIDENCE", "./evidence")

def month_range(start, end):
    (y, m), (ey, em), out = start, end, []
    while (y, m) <= (ey, em):
        out.append((y, m)); m += 1
        if m > 12: y, m = y + 1, 1
    return out

MONTHS = month_range((2022, 1), (2026, 3))       # toàn bộ 51 tháng. Muốn chạy thử: month_range((2022, 1), (2022, 3))
_env = os.environ.get("AERO_MONTHS")
if _env:
    _ym = lambda s: tuple(int(x) for x in s.split("-"))
    MONTHS = [_ym(s) for s in _env.split(",")] if "," in _env else month_range(*[_ym(s) for s in _env.split(":")])
FORCE_REBUILD = False                            # False: tháng nào đã build rồi thì bỏ qua (chạy lại được sau khi bị ngắt)

# ── Quy ước chỉ số (định nghĩa ở MỘT chỗ) ──
LATE_MIN    = 15      # chuyến "trễ" khi ARR_DELAY >= 15 phút (chuẩn DOT/BTS)
MIN_FLIGHTS = 200     # nhóm nhỏ hơn bị loại khi xếp hạng
MIN_BOX_N   = 30      # box-plot của nhóm nhỏ hơn được gắn reliable=False
W_DELAY, W_SEVERE, W_CANCEL, W_DIVERT = 0.50, 0.20, 0.25, 0.05   # trọng số reliability_score (chỉ số tự định nghĩa)

spark = (SparkSession.builder.master("local[2]").appName("usaero-eda")
         .config("spark.driver.memory", "2g")
         .config("spark.sql.shuffle.partitions", os.environ.get("AERO_SHUFFLE", "8"))
         .config("spark.sql.adaptive.enabled", "true").config("spark.sql.session.timeZone", "UTC")
         .config("spark.sql.sources.partitionOverwriteMode", "dynamic").config("spark.sql.warehouse.dir", WAREHOUSE))
if HDFS_URI:
    spark = spark.config("spark.hadoop.fs.defaultFS", HDFS_URI)
spark = spark.getOrCreate()
spark.sparkContext.setLogLevel("WARN")
print("pyspark", pyspark.__version__, "| Spark", spark.version, "| python", sys.version.split()[0],
      f"| {len(MONTHS)} tháng: {MONTHS[0][0]}-{MONTHS[0][1]:02d} → {MONTHS[-1][0]}-{MONTHS[-1][1]:02d}")


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/28 08:27:41 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


pyspark 3.5.9 | Spark 3.5.3 | python 3.8.10 | 51 tháng: 2022-01 → 2026-03


### 2. Nạp dữ liệu theo tháng và chuẩn hóa

In [2]:
# ── Nạp dữ liệu theo tháng: chỉ đọc đúng thư mục year=/month= cần (partition pruning) ──
def _hpath(p):
    path = spark._jvm.org.apache.hadoop.fs.Path(p)
    return path, path.getFileSystem(spark._jsc.hadoopConfiguration())

def month_dir(y, m):
    for name in (f"month={m:02d}", f"month={m}"):
        p = f"{PROCESSED}/year={y}/{name}"
        path, fs = _hpath(p)
        if fs.exists(path): return p
    return None

def load_months(months, verbose=True):
    found = [p for p in (month_dir(y, m) for y, m in months) if p]
    miss = [f"{y}-{m:02d}" for y, m in months if not month_dir(y, m)]
    if miss and verbose: print(f"⚠ Thiếu {len(miss)} tháng (bỏ qua): {', '.join(miss)}")
    if not found: raise FileNotFoundError(f"Không có partition nào dưới {PROCESSED}")
    return spark.read.option("basePath", PROCESSED).parquet(*found)

# ── Tên cột có thể khác nhau: khai báo các tên ứng viên cho mỗi trường logic ──
COLS = {"fl_date": ["FL_DATE", "FlightDate"], "carrier": ["OP_UNIQUE_CARRIER", "OP_CARRIER", "REPORTING_AIRLINE", "CARRIER"],
        "origin": ["ORIGIN"], "dest": ["DEST"], "tail": ["TAIL_NUM"], "crs_dep_time": ["CRS_DEP_TIME"],
        "dep_delay": ["DEP_DELAY"], "arr_delay": ["ARR_DELAY"], "cancelled": ["CANCELLED"], "diverted": ["DIVERTED"],
        "cancel_code": ["CANCELLATION_CODE"], "d_carrier": ["CARRIER_DELAY"], "d_weather": ["WEATHER_DELAY"],
        "d_nas": ["NAS_DELAY"], "d_security": ["SECURITY_DELAY"], "d_late_aircraft": ["LATE_AIRCRAFT_DELAY"]}
REQUIRED = ["fl_date", "carrier", "origin", "dest", "dep_delay", "arr_delay", "cancelled"]
CAUSES = ["carrier", "weather", "nas", "security", "late_aircraft"]
COLMAP = {}

def prepare(raw):
    """Chuẩn hóa MỘT lần ở MỘT chỗ. Hai quy tắc null quan trọng nhất:
       • ARR_DELAY/DEP_DELAY null = chuyến hủy/chuyển hướng → KHÔNG điền 0 (nếu điền, chuyến hủy bị tính là đúng giờ).
       • Cột nguyên nhân trễ null = chuyến không trễ ≥15' → điền 0."""
    global COLMAP
    low = {c.lower(): c for c in raw.columns}
    m = {k: next((low[c.lower()] for c in cands if c.lower() in low), None) for k, cands in COLS.items()}
    miss = [k for k in REQUIRED if m[k] is None]
    if miss: raise ValueError(f"Thiếu cột bắt buộc {miss}. Cột hiện có: {raw.columns}")
    COLMAP = m
    col = lambda k, t="double": F.col(f"`{m[k]}`").cast(t) if m[k] else F.lit(None).cast(t)
    d = raw.select(
        F.to_date(F.col(f"`{m['fl_date']}`")).alias("fl_date"),
        *[F.col(f"`{m[k]}`").cast("string").alias(k) for k in ("carrier", "origin", "dest")],
        col("tail", "string").alias("tail"), col("crs_dep_time", "int").alias("crs_dep_time"),
        col("dep_delay").alias("dep_delay"), col("arr_delay").alias("arr_delay"),
        F.coalesce(col("cancelled", "int"), F.lit(0)).alias("cancelled"),
        F.coalesce(col("diverted", "int"), F.lit(0)).alias("diverted"),
        col("cancel_code", "string").alias("cancel_code"),
        *[F.coalesce(col(f"d_{k}"), F.lit(0.0)).alias(f"d_{k}") for k in CAUSES],
    ).where(F.col("fl_date").isNotNull())
    return (d.withColumn("year", F.year("fl_date")).withColumn("month", F.month("fl_date")).withColumn("day", F.dayofmonth("fl_date"))
             .withColumn("dep_hour", (F.floor(F.col("crs_dep_time") / 100) % 24).cast("int"))
             .withColumn("dep_min", (F.col("crs_dep_time") % 100).cast("int"))
             .withColumn("operated", ((F.col("cancelled") == 0) & (F.col("diverted") == 0)).cast("int"))
             .withColumn("arr_delay_ok", F.when(F.col("operated") == 1, F.col("arr_delay")))     # null nếu hủy/chuyển hướng
             .withColumn("dep_delay_ok", F.when(F.col("cancelled") == 0, F.col("dep_delay")))
             .withColumn("air_gain", F.when(F.col("arr_delay_ok").isNotNull() & F.col("dep_delay_ok").isNotNull(),
                                            F.col("dep_delay_ok") - F.col("arr_delay_ok"))))     # >0: bù được giờ trên không

# ── Bảng tra nhỏ (dùng cho broadcast join) ──
CARRIERS = {"AA": "American", "DL": "Delta", "UA": "United", "WN": "Southwest", "B6": "JetBlue", "AS": "Alaska", "NK": "Spirit",
            "F9": "Frontier", "G4": "Allegiant", "HA": "Hawaiian", "YX": "Republic", "OO": "SkyWest", "MQ": "Envoy", "9E": "Endeavor",
            "OH": "PSA", "YV": "Mesa", "QX": "Horizon", "ZW": "Air Wisconsin", "C5": "CommutAir", "G7": "GoJet", "CP": "Compass",
            "SY": "Sun Country", "MX": "Breeze", "9K": "Cape Air"}
def carrier_lookup(): return spark.createDataFrame(list(CARRIERS.items()), ["carrier", "carrier_name"])

def _nth_weekday(y, m, weekday, n):                 # weekday 0=Thứ hai; n>0: thứ n; n=-1: thứ cuối cùng
    if n > 0:
        d = dt.date(y, m, 1); return d + dt.timedelta(days=(weekday - d.weekday()) % 7 + 7 * (n - 1))
    d = dt.date(y + (m == 12), (m % 12) + 1, 1) - dt.timedelta(days=1)
    return d - dt.timedelta(days=(d.weekday() - weekday) % 7)

def holiday_df(years=range(2021, 2028), window=3):
    """Ngày trong ±3 ngày quanh 6 kỳ nghỉ lớn của Mỹ."""
    best = {}
    for y in years:
        hol = {"new_year": dt.date(y, 1, 1), "memorial": _nth_weekday(y, 5, 0, -1), "july4": dt.date(y, 7, 4),
               "labor": _nth_weekday(y, 9, 0, 1), "thanksgiving": _nth_weekday(y, 11, 3, 4), "christmas": dt.date(y, 12, 25)}
        for name, d0 in hol.items():
            for off in range(-window, window + 1):
                d = d0 + dt.timedelta(days=off)
                if d not in best or abs(off) < abs(best[d][1]): best[d] = (name, off)
    sch = T.StructType([T.StructField("fl_date", T.DateType()), T.StructField("holiday", T.StringType()), T.StructField("holiday_offset", T.IntegerType())])
    return spark.createDataFrame([(d, n, o) for d, (n, o) in best.items()], sch)

def peek(df, n=15):
    try:
        pdf = df.limit(n).toPandas()
        try: display(pdf)
        except NameError: print(pdf.to_string())
    except Exception:
        df.show(n, truncate=False)

def timed(fn):
    t = time.time(); r = fn(); return r, round(time.time() - t, 2)


### 3. Lõi KPI: cột cộng được → cuộn lên → tỷ lệ

In [3]:
# ── Ý TƯỞNG 1: chỉ tính các cột CỘNG ĐƯỢC (đếm, tổng) một lần cho mỗi tháng. Mọi mức ngày/tháng/năm là phép CỘNG lại;
#    tỷ lệ luôn tính từ tổng (không lấy trung bình của các tỷ lệ). Nhờ vậy các mức khớp nhau và thêm 1 tháng chỉ tốn 1 tháng. ──
def _add_exprs():
    a, dp = F.col("arr_delay_ok"), F.col("dep_delay_ok")
    cnt = lambda cond: F.sum(F.when(cond, 1).otherwise(0))
    E = [("flights", F.count(F.lit(1))), ("cancelled", F.sum("cancelled")), ("diverted", F.sum("diverted")),
         ("arr_n", F.count(a)), ("on_time", cnt(a < LATE_MIN)),
         ("delayed15", cnt(a >= 15)), ("delayed60", cnt(a >= 60)),
         ("sum_arr_delay", F.sum(a)), ("sum_arr_delay_pos", F.sum(F.when(a > 0, a).otherwise(0.0))),
         ("sum_arr_delay_late", F.sum(F.when(a >= LATE_MIN, a).otherwise(0.0))),
         ("dep_n", F.count(dp)), ("dep_delayed15", cnt(dp >= 15)), ("sum_dep_delay_pos", F.sum(F.when(dp > 0, dp).otherwise(0.0))),
         ("gain_n", F.count("air_gain")), ("sum_air_gain", F.sum("air_gain"))]
    E += [(f"cause_{k}", F.sum(f"d_{k}")) for k in CAUSES]
    E += [(f"cancel_{k}", cnt(F.col("cancel_code") == k.upper())) for k in "abcd"]       # A=hãng, B=thời tiết, C=NAS, D=an ninh
    return E

ADD_COLS = [n for n, _ in _add_exprs()]

def kpi_base(df, keys):
    return df.groupBy(*keys).agg(*[e.alias(n) for n, e in _add_exprs()])

def rollup(base, keys):
    """Cộng các cột cộng được theo khóa thô hơn. n_days = số ngày có dữ liệu (để tính chuyến/ngày, nhận biết năm chưa đủ)."""
    cols = set(base.columns)
    if "n_days" in cols: nd = [F.sum("n_days").alias("n_days")]
    elif {"year", "month", "day"} <= cols: nd = [F.countDistinct("year", "month", "day").alias("n_days")]
    else: nd = []
    return base.groupBy(*keys).agg(*nd, *[F.sum(c).alias(c) for c in ADD_COLS])

def _r(num, den):
    return F.when(F.col(den) > 0, F.col(num) / F.col(den))

def finalize(df):
    rates = {"cancel_rate": _r("cancelled", "flights"), "divert_rate": _r("diverted", "flights"),
             "otp_operated": _r("on_time", "arr_n"),       # đúng giờ trên các chuyến đã đo được giờ đến
             "otp_scheduled": _r("on_time", "flights"),    # đúng giờ trên TẤT CẢ chuyến lên lịch (hủy/chuyển hướng = không đúng giờ)
             "delay15_rate": _r("delayed15", "arr_n"), "severe60_rate": _r("delayed60", "arr_n"),
             "avg_arr_delay_net": _r("sum_arr_delay", "arr_n"),        # giữ số âm (đến sớm bù trừ)
             "avg_arr_delay_pos": _r("sum_arr_delay_pos", "arr_n"),    # chỉ tính phần trễ
             "avg_delay_when_late": _r("sum_arr_delay_late", "delayed15"),
             "dep_delay15_rate": _r("dep_delayed15", "dep_n"), "avg_dep_delay_pos": _r("sum_dep_delay_pos", "dep_n"),
             "avg_air_gain": _r("sum_air_gain", "gain_n"),
             **{f"cancel_share_{n}": _r(f"cancel_{k}", "cancelled") for k, n in zip("abcd", ("carrier", "weather", "nas", "security"))}}
    out = df.select("*", *[v.alias(k) for k, v in rates.items()])
    cc = [f"cause_{k}" for k in CAUSES]
    out = out.withColumn("cause_total", sum(F.coalesce(F.col(c), F.lit(0.0)) for c in cc))
    out = out.select("*", *[F.when(F.col("cause_total") > 0, F.col(c) / F.col("cause_total")).alias(c + "_share") for c in cc])
    z = lambda c: F.coalesce(F.col(c), F.lit(0.0))
    pen = W_DELAY * z("delay15_rate") + W_SEVERE * z("severe60_rate") + W_CANCEL * z("cancel_rate") + W_DIVERT * z("divert_rate")
    return out.withColumn("reliability_score", F.greatest(F.lit(0.0), F.least(F.lit(100.0), 100 * (1 - pen))))   # chỉ số tự định nghĩa 0-100


### 4. Box-plot từ biểu đồ tần suất

In [4]:
# ── Ý TƯỞNG 2: box-plot CHÍNH XÁC từ biểu đồ tần suất cộng được. ──
# Phân vị không cộng được, nhưng "số chuyến có delay = v phút" thì cộng được. BTS ghi delay là số phút nguyên nên biểu đồ tần suất
# (mỗi giá trị v một dòng) giữ nguyên thông tin: tính một lần mỗi tháng, mọi mức ngày/tháng/năm chỉ cần cộng lại rồi đọc phân vị.
def build_hist(df):
    one = lambda var, c: df.select("year", "month", "day", "carrier", F.lit(var).alias("var"), F.round(c).cast("int").alias("v"))
    return (one("arr", "arr_delay_ok").unionByName(one("dep", "dep_delay_ok")).where(F.col("v").isNotNull())
            .groupBy("year", "month", "day", "carrier", "var", "v").agg(F.count(F.lit(1)).alias("n")))

def box_from_hist(hist, keys, population):
    """Thống kê box-plot theo `keys`: population 'all' (mọi chuyến đã bay) hoặc 'late' (chỉ chuyến trễ ≥15 phút)."""
    g = keys + ["var"]
    h = hist if population == "all" else hist.where(F.col("v") >= LATE_MIN)
    d = h.groupBy(*g, "v").agg(F.sum("n").alias("n"))
    d = (d.withColumn("cum", F.sum("n").over(Window.partitionBy(*g).orderBy("v")))       # tần suất tích lũy
          .withColumn("total", F.sum("n").over(Window.partitionBy(*g))))
    q = lambda p: F.min(F.when(F.col("cum") >= F.greatest(F.lit(1), F.ceil(F.col("total") * p - 1e-9)), F.col("v")))   # phân vị theo hạng gần nhất
    st = (d.groupBy(*g).agg(F.max("total").alias("n"), F.min("v").alias("min"), F.max("v").alias("max"),
                            F.sum(F.col("v") * F.col("n")).alias("_s1"), F.sum(F.col("v") * F.col("v") * F.col("n")).alias("_s2"),
                            q(0.05).alias("p05"), q(0.25).alias("q1"), q(0.50).alias("median"), q(0.75).alias("q3"),
                            q(0.90).alias("p90"), q(0.95).alias("p95"), q(0.99).alias("p99"))
            .withColumn("mean", F.col("_s1") / F.col("n"))
            .withColumn("std", F.sqrt(F.greatest(F.lit(0.0), F.col("_s2") / F.col("n") - F.col("mean") ** 2))).drop("_s1", "_s2")
            .withColumn("iqr", F.col("q3") - F.col("q1"))
            .withColumn("fence_low", F.col("q1") - 1.5 * F.col("iqr")).withColumn("fence_high", F.col("q3") + 1.5 * F.col("iqr")))
    # Lượt 2: broadcast join hàng rào Tukey (bảng rất nhỏ) ngược vào biểu đồ → đầu râu thật + số ngoại lệ
    w = (d.join(F.broadcast(st.select(*g, "fence_low", "fence_high")), g).groupBy(*g).agg(
            F.min(F.when(F.col("v") >= F.col("fence_low"), F.col("v"))).alias("whisker_low"),
            F.max(F.when(F.col("v") <= F.col("fence_high"), F.col("v"))).alias("whisker_high"),
            F.sum(F.when(F.col("v") < F.col("fence_low"), F.col("n")).otherwise(0)).alias("n_out_low"),
            F.sum(F.when(F.col("v") > F.col("fence_high"), F.col("n")).otherwise(0)).alias("n_out_high")))
    return (st.join(w, g).withColumn("population", F.lit(population))
              .withColumn("out_high_rate", F.col("n_out_high") / F.col("n")).withColumn("reliable", F.col("n") >= MIN_BOX_N))


### 5. Pipeline: mỗi tháng một lần, rồi cuộn lên mọi mức

In [5]:
# ── Pipeline: mỗi tháng đọc dữ liệu thô MỘT lần → 5 bảng KPI cơ sở + 1 biểu đồ tần suất. Mọi thứ còn lại chỉ cộng bảng nhỏ. ──
GRAINS = {"day": ["year", "month", "day"],        # chuỗi ngày: xem 1 tháng (hướng A) hoặc cả năm (hướng B)
          "calendar_day": ["month", "day"],       # cùng ngày lịch qua các năm (25/12 của mọi năm)
          "month": ["year", "month"],             # các tháng trong 1 năm / chuỗi tháng
          "month_of_year": ["month"],             # cùng một tháng qua các năm → mùa vụ
          "year": ["year"]}
BASE_SPECS = {"day_carrier": ["year", "month", "day", "carrier"], "day_origin": ["year", "month", "day", "origin"],
              "month_carrier_origin": ["year", "month", "carrier", "origin"], "month_route": ["year", "month", "origin", "dest"],
              "prof_hour": ["year", "month", "dep_hour"]}

def write_part(df, sub):
    df.coalesce(1).write.mode("overwrite").partitionBy("year", "month").parquet(f"{ANALYTICS}/{sub}")

def month_built(y, m):
    path, fs = _hpath(f"{ANALYTICS}/base/hist/year={y}/month={m}")       # hist được ghi cuối cùng → có nghĩa là tháng đã xong
    return fs.exists(path)

def build_month(y, m):
    t0 = time.time()
    df = prepare(load_months([(y, m)], verbose=False)).persist(StorageLevel.MEMORY_AND_DISK)
    n = df.count()
    for name, keys in BASE_SPECS.items():
        write_part(kpi_base(df, keys), f"base/{name}")
    write_part(build_hist(df), "base/hist")
    df.unpersist()
    return n, round(time.time() - t0, 1)

def build_months(months):
    todo = [(y, m) for y, m in months if month_dir(y, m) and (FORCE_REBUILD or not month_built(y, m))]
    skip = [f"{y}-{m:02d}" for y, m in months if month_dir(y, m) and (y, m) not in todo]
    gone = [f"{y}-{m:02d}" for y, m in months if not month_dir(y, m)]
    if gone: print(f"⚠ Không có dữ liệu: {', '.join(gone)}")
    if skip: print(f"↷ Đã build trước đó (bỏ qua {len(skip)} tháng; đặt FORCE_REBUILD=True để tính lại)")
    for y, m in todo:
        n, sec = build_month(y, m)
        print(f"  {y}-{m:02d}: {n:>9,} dòng | {sec:>6}s", flush=True)

# Cuộn lên: (tên bảng, bảng cơ sở, khóa nhóm)
KPI_PLAN = [("day_overall", "day_carrier", GRAINS["day"]), ("calendar_day_overall", "day_carrier", GRAINS["calendar_day"]),
            ("month_overall", "day_carrier", GRAINS["month"]), ("month_of_year_overall", "day_carrier", GRAINS["month_of_year"]),
            ("year_overall", "day_carrier", GRAINS["year"]),
            ("month_carrier", "day_carrier", GRAINS["month"] + ["carrier"]), ("month_of_year_carrier", "day_carrier", GRAINS["month_of_year"] + ["carrier"]),
            ("year_carrier", "day_carrier", GRAINS["year"] + ["carrier"]),
            ("month_origin", "day_origin", GRAINS["month"] + ["origin"]), ("year_origin", "day_origin", GRAINS["year"] + ["origin"]),
            ("year_route", "month_route", GRAINS["year"] + ["origin", "dest"]),
            ("year_carrier_origin", "month_carrier_origin", GRAINS["year"] + ["carrier", "origin"]),
            ("hour_overall", "prof_hour", ["dep_hour"])]
BOX_PLAN = [("day", []), ("month", []), ("month", ["carrier"]), ("month_of_year", []), ("month_of_year", ["carrier"]), ("year", []), ("year", ["carrier"])]

def build_kpis():
    for name, base, keys in KPI_PLAN:
        b = spark.read.parquet(f"{ANALYTICS}/base/{base}")
        finalize(rollup(b, keys)).coalesce(1).write.mode("overwrite").parquet(f"{ANALYTICS}/kpi/{name}")

def build_boxes():
    hist = spark.read.parquet(f"{ANALYTICS}/base/hist")
    for grain, extra in BOX_PLAN:
        keys = GRAINS[grain] + extra
        out = box_from_hist(hist, keys, "all").unionByName(box_from_hist(hist, keys, "late"))
        out.coalesce(1).write.mode("overwrite").parquet(f"{ANALYTICS}/box/{grain}_{'carrier' if extra else 'overall'}")

def kpi(name):      return spark.read.parquet(f"{ANALYTICS}/kpi/{name}")
def box(name):      return spark.read.parquet(f"{ANALYTICS}/box/{name}")
def base_tbl(name): return spark.read.parquet(f"{ANALYTICS}/base/{name}")


### 6. Chạy pipeline

In [6]:
# ═══════════ CHẠY PIPELINE ═══════════
t0 = time.time()
print("Bước 1/3 – từng tháng (đọc dữ liệu thô một lần):");  build_months(MONTHS)
print("Bước 2/3 – cuộn lên các mức KPI…");                  build_kpis()
print("Bước 3/3 – box-plot từ biểu đồ tần suất…");           build_boxes()
print(f"Hoàn tất sau {round(time.time() - t0, 1)}s → {ANALYTICS}")


Bước 1/3 – từng tháng (đọc dữ liệu thô một lần):
↷ Đã build trước đó (bỏ qua 2 tháng; đặt FORCE_REBUILD=True để tính lại)


26/10/28 08:28:13 WARN DataSource: [COLUMN_ALREADY_EXISTS] The column `month` already exists. Consider to choose another name or rename the existing column.
26/10/28 08:28:29 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


  2022-03:   564,853 dòng |   33.4s


26/10/28 08:28:44 WARN DataSource: [COLUMN_ALREADY_EXISTS] The column `month` already exists. Consider to choose another name or rename the existing column.


  2022-04:   556,502 dòng |   13.9s


...

26/10/28 08:36:06 WARN DataSource: [COLUMN_ALREADY_EXISTS] The column `month` already exists. Consider to choose another name or rename the existing column.


  2026-03:   612,102 dòng |   12.9s


Bước 2/3 – cuộn lên các mức KPI…


Bước 3/3 – box-plot từ biểu đồ tần suất…


Hoàn tất sau 592.9s → /aero/analytics/eda


### 7. Các hàm phân tích

In [7]:
# ═══════════ CÁC PHÂN TÍCH (đọc từ bảng KPI nhỏ đã tính; mỗi hàm trả về một DataFrame nhỏ) ═══════════

# ── Tổng quan và MỨC NĂM ──
def overview():
    return (kpi("year_overall").withColumn("flights_per_day", F.col("flights") / F.col("n_days")).orderBy("year")
            .select("year", "n_days", "flights", "flights_per_day", "cancel_rate", "divert_rate", "otp_operated", "otp_scheduled",
                    "avg_arr_delay_net", "avg_arr_delay_pos", "avg_delay_when_late", "delay15_rate", "severe60_rate",
                    "avg_air_gain", "reliability_score"))

# ── MỨC THÁNG ──
def monthly_trend(metric="otp_scheduled"):
    """Chuỗi tháng + trung bình trượt 3 tháng (window) + so cùng kỳ năm trước (self-join t-12)."""
    cur = kpi("month_overall").withColumn("t", F.col("year") * 12 + F.col("month")).select("year", "month", "t", "flights", metric)
    prev = cur.select((F.col("t") + 12).alias("t"), F.col(metric).alias(f"{metric}_prev_year"))
    return (cur.join(prev, "t", "left").withColumn(f"{metric}_ma3", F.avg(metric).over(Window.orderBy("t").rangeBetween(-2, 0)))
               .withColumn("yoy_change", F.col(metric) - F.col(f"{metric}_prev_year")).orderBy("t").drop("t"))

def seasonality(metric="delay15_rate"):
    """Chỉ số mùa vụ = giá trị của tháng-trong-năm / giá trị chung (>1: tháng xấu hơn bình thường nếu là chỉ số 'xấu')."""
    g = finalize(rollup(kpi("year_overall"), [])).select(F.col(metric).alias("overall_value"))
    return (kpi("month_of_year_overall").select("month", "flights", metric).crossJoin(g)
            .withColumn("seasonality_index", F.col(metric) / F.col("overall_value")).orderBy("month"))

def month_matrix(metric="otp_scheduled"):
    """PIVOT: hàng = tháng, cột = năm → so cùng tháng qua các năm."""
    k = kpi("month_overall"); years = sorted(r[0] for r in k.select("year").distinct().collect())
    return k.groupBy("month").pivot("year", years).agg(F.first(metric)).orderBy("month")

# ── MỨC NGÀY ──
def day_in_month(y, m):
    """Hướng A: các ngày của MỘT tháng, kèm xếp hạng ngày xấu nhất."""
    k = kpi("day_overall").where((F.col("year") == y) & (F.col("month") == m))
    return (k.withColumn("dow", ((F.dayofweek(F.make_date("year", "month", "day")) + 5) % 7) + 1)      # 1=Thứ hai … 7=Chủ nhật
             .withColumn("rank_worst", F.dense_rank().over(Window.orderBy(F.desc("delay15_rate")))).orderBy("day")
             .select("day", "dow", "flights", "otp_scheduled", "cancel_rate", "delay15_rate", "avg_arr_delay_pos", "rank_worst"))

def daily_anomalies(metric="delay15_rate", window_days=28, top=10):
    """Hướng B: z-score từng ngày so với nền 28 ngày trước (window theo khoảng ngày); gắn ngày lễ (broadcast join)."""
    d = kpi("day_overall").withColumn("d", F.make_date("year", "month", "day")).withColumn("dn", F.datediff("d", F.lit("1970-01-01").cast("date")))
    w = Window.orderBy("dn").rangeBetween(-window_days, -1)
    hol = holiday_df().select(F.col("fl_date").alias("d"), "holiday", "holiday_offset")
    z = (d.withColumn("z", (F.col(metric) - F.avg(metric).over(w)) / F.stddev(metric).over(w))
          .join(F.broadcast(hol), "d", "left").withColumn("holiday", F.coalesce("holiday", F.lit("none"))))
    return z.orderBy(F.desc(F.abs("z"))).limit(top).select("d", "flights", metric, "z", "cancel_rate", "avg_arr_delay_pos", "holiday", "holiday_offset")

def calendar_day_across_years(top=10):
    """Hướng B: cùng một ngày lịch gộp qua các năm."""
    return kpi("calendar_day_overall").orderBy(F.desc("delay15_rate")).limit(top).select("month", "day", "n_days", "flights", "delay15_rate", "cancel_rate", "otp_scheduled")

def _with_calendar():
    d = kpi("day_overall").withColumn("fl_date", F.make_date("year", "month", "day"))
    return d.withColumn("dow", ((F.dayofweek("fl_date") + 5) % 7) + 1)

def dow_profile():
    return finalize(rollup(_with_calendar(), ["dow"])).orderBy("dow").select("dow", "flights", "otp_scheduled", "delay15_rate", "cancel_rate", "avg_arr_delay_pos")

def holiday_effect():
    """Mức nâng (lift) so với ngày thường theo số ngày cách lễ (broadcast join bảng ngày lễ)."""
    d = _with_calendar().join(F.broadcast(holiday_df()), "fl_date", "left")
    d = d.withColumn("holiday", F.coalesce("holiday", F.lit("none"))).withColumn("holiday_offset", F.coalesce("holiday_offset", F.lit(0)))
    h = finalize(rollup(d, ["holiday", "holiday_offset"]))
    b = h.where(F.col("holiday") == "none").select(F.col("delay15_rate").alias("b_delay"), F.col("cancel_rate").alias("b_cancel"))
    return (h.where(F.col("holiday") != "none").crossJoin(b).withColumn("delay15_lift", F.col("delay15_rate") / F.col("b_delay"))
             .withColumn("cancel_lift", F.col("cancel_rate") / F.col("b_cancel")).orderBy("holiday", "holiday_offset")
             .select("holiday", "holiday_offset", "flights", "delay15_rate", "delay15_lift", "cancel_rate", "cancel_lift"))

def hour_profile():
    """Trễ tích lũy theo giờ khởi hành: chuyến sáng sớm ít trễ, chiều tối trễ nhiều (hiệu ứng máy bay đến trễ)."""
    return kpi("hour_overall").orderBy("dep_hour").select("dep_hour", "flights", "otp_scheduled", "delay15_rate", "avg_dep_delay_pos",
                                                         "cause_late_aircraft_share", "cancel_rate")

# ── HÃNG BAY ──
def carrier_ranking(year=None):
    k = kpi("year_carrier").where(F.col("flights") >= MIN_FLIGHTS).join(F.broadcast(carrier_lookup()), "carrier", "left")
    if year: k = k.where(F.col("year") == year)
    return (k.withColumn("rank", F.dense_rank().over(Window.partitionBy("year").orderBy(F.desc("reliability_score")))).orderBy("year", "rank")
             .select("year", "rank", "carrier", "carrier_name", "flights", "otp_scheduled", "cancel_rate", "delay15_rate",
                     "avg_delay_when_late", "severe60_rate", "reliability_score"))

def carrier_yoy():
    """Hãng nào cải thiện / xấu đi so với năm trước (self-join theo year-1)."""
    k = kpi("year_carrier").where(F.col("flights") >= MIN_FLIGHTS)
    p = k.select((F.col("year") + 1).alias("year"), "carrier", F.col("otp_scheduled").alias("otp_prev"), F.col("cancel_rate").alias("cancel_prev"))
    return (k.join(p, ["year", "carrier"]).withColumn("otp_change", F.col("otp_scheduled") - F.col("otp_prev"))
             .withColumn("cancel_change", F.col("cancel_rate") - F.col("cancel_prev")).orderBy("year", F.desc("otp_change"))
             .select("year", "carrier", "flights", "otp_prev", "otp_scheduled", "otp_change", "cancel_prev", "cancel_rate", "cancel_change"))

def carrier_matrix(metric="otp_scheduled", by="year"):
    """PIVOT: hàng = hãng, cột = năm (hoặc tháng-trong-năm)."""
    k = kpi(f"{by}_carrier"); col = "month" if by == "month_of_year" else "year"
    vals = sorted(r[0] for r in k.select(col).distinct().collect())
    return k.where(F.col("flights") >= MIN_FLIGHTS).groupBy("carrier").pivot(col, vals).agg(F.first(metric)).orderBy("carrier")

def unpivot_causes(df, keys):
    """UNPIVOT: từ 5 cột nguyên nhân (dạng rộng) sang dạng dài (cause, minutes, share)."""
    stack = "stack(5, 'carrier', cause_carrier, 'weather', cause_weather, 'nas', cause_nas, 'security', cause_security, 'late_aircraft', cause_late_aircraft) as (cause, minutes)"
    return (df.select(*keys, "cause_total", F.expr(stack))
              .withColumn("share", F.when(F.col("cause_total") > 0, F.col("minutes") / F.col("cause_total"))))

# ── SÂN BAY, MẠNG LƯỚI, TUYẾN ──
def worst_airports(year=None, top=10):
    k = kpi("year_origin").where(F.col("flights") >= MIN_FLIGHTS)
    if year: k = k.where(F.col("year") == year)
    return k.orderBy(F.desc("avg_dep_delay_pos")).limit(top).select("year", "origin", "flights", "otp_scheduled", "cancel_rate", "avg_dep_delay_pos", "cause_late_aircraft_share")

def concentration():
    """HHI thị phần hãng tại sân bay đi (>0.25 = tập trung cao, theo ngưỡng DOJ 2.500/10.000)."""
    co = kpi("year_carrier_origin").select("year", "origin", "carrier", "flights")
    tot = co.groupBy("year", "origin").agg(F.sum("flights").alias("airport_flights"))
    j = co.join(tot, ["year", "origin"]).withColumn("share", F.col("flights") / F.col("airport_flights"))
    top = (j.withColumn("rk", F.row_number().over(Window.partitionBy("year", "origin").orderBy(F.desc("share")))).where("rk = 1")
             .select("year", "origin", F.col("carrier").alias("dominant_carrier"), F.col("share").alias("dominant_share")))
    hhi = j.groupBy("year", "origin", "airport_flights").agg(F.sum(F.col("share") ** 2).alias("hhi"))
    return (hhi.join(top, ["year", "origin"]).where(F.col("airport_flights") >= MIN_FLIGHTS)
               .withColumn("hhi_class", F.when(F.col("hhi") > 0.25, "concentrated").when(F.col("hhi") > 0.15, "moderate").otherwise("competitive")))

def _pagerank(pdf, d=0.85, iters=60):
    import numpy as np, pandas as pd
    nodes = sorted(set(pdf["origin"]) | set(pdf["dest"])); idx = {n: i for i, n in enumerate(nodes)}; n = len(nodes)
    src, dst, w = pdf["origin"].map(idx).values, pdf["dest"].map(idx).values, pdf["flights"].values.astype(float)
    out = np.zeros(n); np.add.at(out, src, w); pr = np.full(n, 1.0 / n)
    for _ in range(iters):
        c = np.zeros(n); np.add.at(c, dst, pr[src] * w / out[src]); pr = (1 - d) / n + d * (c + pr[out == 0].sum() / n)
    return pd.DataFrame({"airport": nodes, "pagerank": pr})

def airport_network(year=None, top=15):
    """Đồ thị sân bay (cạnh = tuyến có hướng, trọng số = số chuyến): bậc vào/ra, lưu lượng, mất cân bằng, PageRank (đồ thị nhỏ → tính ở driver)."""
    e = kpi("year_route")
    if year: e = e.where(F.col("year") == year)
    e = e.groupBy("origin", "dest").agg(F.sum("flights").alias("flights"))
    o = e.groupBy(F.col("origin").alias("airport")).agg(F.count("dest").alias("out_degree"), F.sum("flights").alias("flights_out"))
    i = e.groupBy(F.col("dest").alias("airport")).agg(F.count("origin").alias("in_degree"), F.sum("flights").alias("flights_in"))
    nodes = o.join(i, "airport", "outer").na.fill(0).withColumn("flow_balance", F.col("flights_out") - F.col("flights_in"))
    pr = spark.createDataFrame(_pagerank(e.select("origin", "dest", "flights").toPandas()))
    return nodes.join(pr, "airport", "left").orderBy(F.desc("pagerank")).limit(top)

def route_excess_delay(year=None, hint="merge", top=10):
    """Tuyến nào trễ NHIỀU HƠN mức nền của sân bay đi? Ghép hai bảng lớn bằng sort-merge join (hint='merge') hoặc broadcast."""
    r = kpi("year_route").where(F.col("flights") >= MIN_FLIGHTS).select("year", "origin", "dest", "flights", F.col("avg_arr_delay_net").alias("route_delay"))
    o = kpi("year_origin").select("year", "origin", F.col("avg_arr_delay_net").alias("origin_baseline"))
    if year: r, o = r.where(F.col("year") == year), o.where(F.col("year") == year)
    return (r.hint(hint).join(o.hint(hint), ["year", "origin"]).withColumn("excess_delay", F.col("route_delay") - F.col("origin_baseline"))
             .orderBy(F.desc("excess_delay")))

def enrich_routes(year=None):
    """NHIỀU JOIN liên tiếp (tuyến ⨝ nhóm sân bay đi ⨝ nhóm sân bay đến ⨝ mức nền sân bay đi), có kiểm tra số dòng không bị nhân bản."""
    r = kpi("year_route")
    if year: r = r.where(F.col("year") == year)
    n0 = r.count()
    o = kpi("year_origin").groupBy("origin").agg(F.sum("flights").alias("f"))
    tier = o.withColumn("airport_tier", F.when(F.row_number().over(Window.orderBy(F.desc("f"))) <= 30, "large").otherwise("other")).select("origin", "airport_tier")
    base = kpi("year_origin").select("year", "origin", F.col("avg_arr_delay_net").alias("origin_baseline"))
    j = (r.join(F.broadcast(tier.withColumnRenamed("airport_tier", "origin_tier")), "origin")
          .join(F.broadcast(tier.withColumnRenamed("origin", "dest").withColumnRenamed("airport_tier", "dest_tier")), "dest")
          .join(base, ["year", "origin"], "left"))
    n1 = j.count()
    print(f"enrich_routes: {n0} dòng trước join, {n1} sau join → {'OK, không nhân bản' if n0 == n1 else 'CẢNH BÁO: số dòng thay đổi!'}")
    return j.select("year", "origin", "dest", F.concat_ws("→", "origin_tier", "dest_tier").alias("tier_pair"), "flights", "otp_scheduled", "avg_arr_delay_net", "origin_baseline")

# ── BOX-PLOT ──
def box_summary(grain="year", dim="overall", var="arr", population="late", where=None):
    b = box(f"{grain}_{dim}").where((F.col("var") == var) & (F.col("population") == population) & F.col("reliable"))
    if where is not None: b = b.where(where)
    keys = [c for c in b.columns if c in ("year", "month", "day", "carrier")]
    return b.select(*keys, "n", "min", "q1", "median", "q3", "whisker_high", "p90", "p95", "p99", "max", "mean", "n_out_high", "out_high_rate").orderBy(*keys)


### 8. Xem kết quả

In [8]:
# ═══════════ XEM KẾT QUẢ ═══════════
def section(t): print("\n" + "═" * 8 + " " + t + " " + "═" * 8)
y0, m0 = MONTHS[0]

section("Tổng quan theo năm (n_days = số ngày có dữ liệu; năm cuối có thể chưa đủ)"); peek(overview())
section("MỨC NĂM – xếp hạng hãng theo reliability_score và thay đổi so với năm trước"); peek(carrier_ranking(), 40); peek(carrier_yoy(), 40)
section("MỨC THÁNG – chuỗi tháng + trung bình trượt + so cùng kỳ"); peek(monthly_trend(), 60)
section("MỨC THÁNG – mùa vụ (tháng-trong-năm so với toàn kỳ) và ma trận tháng × năm"); peek(seasonality(), 12); peek(month_matrix(), 12)
section(f"MỨC NGÀY – hướng A: các ngày của tháng {y0}-{m0:02d}"); peek(day_in_month(y0, m0), 31)
section("MỨC NGÀY – hướng B: ngày bất thường và ngày lịch xấu nhất qua các năm"); peek(daily_anomalies()); peek(calendar_day_across_years())
section("HỒ SƠ – theo giờ khởi hành / thứ trong tuần / ngày lễ"); peek(hour_profile(), 24); peek(dow_profile(), 7); peek(holiday_effect(), 50)
section("NGUYÊN NHÂN TRỄ – dạng dài (unpivot) theo năm"); peek(unpivot_causes(kpi("year_overall"), ["year"]), 20)
section("HÃNG – ma trận hãng × năm (pivot)"); peek(carrier_matrix("otp_scheduled", "year"), 40)
section("SÂN BAY – sân bay trễ nhiều nhất và mức tập trung hãng (HHI)"); peek(worst_airports()); peek(concentration().orderBy(F.desc("hhi")), 10)
section("MẠNG LƯỚI – bậc vào/ra, lưu lượng, PageRank"); peek(airport_network())
section("TUYẾN – trễ vượt mức nền sân bay đi (sort-merge join) và bảng tuyến nhiều join"); peek(route_excess_delay()); peek(enrich_routes(), 10)
section("BOX-PLOT – delay đến toàn bộ chuyến đã bay, theo năm"); peek(box_summary("year", "overall", "arr", "all"), 10)
section("BOX-PLOT – delay đến của chuyến TRỄ, theo hãng và năm"); peek(box_summary("year", "carrier", "arr", "late"), 40)



════════ Tổng quan theo năm (n_days = số ngày có dữ liệu; năm cuối có thể chưa đủ) ════════


,year,n_days,flights,flights_per_day,cancel_rate,divert_rate,otp_operated,otp_scheduled,avg_arr_delay_net,avg_arr_delay_pos,avg_delay_when_late,delay15_rate,severe60_rate,avg_air_gain,reliability_score
0,2022,365,6729125,18435.958904,0.026941,0.002352,0.789223,0.766105,7.035647,15.233843,67.283050,0.210777,0.070074,5.493606,87.374401
1,2023,365,6847899,18761.367123,0.012842,0.002417,0.794362,0.782241,6.627152,15.231402,69.011667,0.205638,0.068507,5.573830,88.014827
2,2024,366,7079061,19341.696721,0.013606,0.002472,0.791828,0.779098,7.098219,15.896336,71.584450,0.208172,0.073455,5.490939,87.769822
3,2025,365,7001619,19182.517808,0.014693,0.002751,0.776925,0.763373,8.504504,17.266110,72.941107,0.223075,0.081773,4.957091,86.829735
4,2026,90,1671142,18568.244444,0.032915,0.002550,0.782392,0.754644,7.262870,17.248902,74.892455,0.217608,0.079642,6.642932,86.691116



════════ MỨC NĂM – xếp hạng hãng theo reliability_score và thay đổi so với năm trước ════════


,year,rank,carrier,carrier_name,flights,otp_scheduled,cancel_rate,delay15_rate,avg_delay_when_late,severe60_rate,reliability_score
0,2022,1,DL,Delta,893172,0.825301,0.016029,0.159664,69.284329,0.051274,90.581278
1,2022,2,QX,Horizon,88791,0.817921,0.017029,0.166401,52.558943,0.043847,90.368405
2,2022,3,OO,SkyWest,733576,0.816097,0.018173,0.166488,81.366734,0.062379,89.960079
3,2022,4,MQ,Envoy,248059,0.806453,0.022374,0.173147,60.844262,0.053704,89.697748
4,2022,5,9E,Endeavor,231941,0.805265,0.035362,0.163375,79.397582,0.062406,89.688454
5,2022,6,UA,United,626473,0.790092,0.018973,0.192529,71.179057,0.069238,88.501731
6,2022,7,AS,Alaska,228384,0.772769,0.027226,0.203182,51.271160,0.049934,88.146781
7,2022,8,OH,PSA,214450,0.779608,0.030832,0.193339,72.763894,0.072620,88.096338
8,2022,9,YX,Republic,316524,0.764245,0.046954,0.195941,68.281701,0.071640,87.583500
9,2022,10,YV,Mesa,114779,0.774271,0.033194,0.196946,90.857903,0.086631,87.576998


,year,carrier,flights,otp_prev,otp_scheduled,otp_change,cancel_prev,cancel_rate,cancel_change
0,2023,G4,115539,0.633940,0.733709,0.099769,0.035165,0.006768,-0.028397
1,2023,YX,295275,0.764245,0.843465,0.079219,0.046954,0.027645,-0.019308
2,2023,OH,194144,0.779608,0.820638,0.041030,0.030832,0.013243,-0.017590
3,2023,WN,1438465,0.731829,0.772542,0.040713,0.032632,0.009959,-0.022674
4,2023,9E,201520,0.805265,0.841118,0.035852,0.035362,0.021050,-0.014312
5,2023,B6,274852,0.646343,0.670779,0.024437,0.037424,0.020968,-0.016457
6,2023,AS,245344,0.772769,0.794782,0.022013,0.027226,0.008058,-0.019168
7,2023,AA,940531,0.749378,0.759367,0.009989,0.029622,0.010609,-0.019013
8,2023,F9,177542,0.660945,0.669892,0.008948,0.028653,0.021257,-0.007396
9,2023,OO,675163,0.816097,0.824632,0.008535,0.018173,0.012222,-0.005950


...

,year,origin,dest,tier_pair,flights,otp_scheduled,avg_arr_delay_net,origin_baseline
0,2024,MCO,DTW,large→large,3798,0.748815,11.663276,10.650136
1,2024,DCA,BOS,large→large,9658,0.787637,5.819648,5.559043
2,2024,DFW,GRR,large→other,1125,0.649778,23.806685,13.908512
3,2024,MSP,DTW,large→large,2327,0.846584,1.405440,4.043018
4,2024,PHL,CHS,large→other,1374,0.781659,7.582046,8.535498
5,2024,ATL,SAT,large→other,2986,0.793034,5.058804,6.824339
6,2024,PVD,CLT,other→large,1584,0.780303,12.308184,4.876391
7,2024,MDW,DTW,large→large,2083,0.786846,6.559051,6.238456
8,2024,GRR,CLT,other→large,1067,0.773196,11.632227,5.561234
9,2024,RSW,BOS,other→large,2219,0.707075,15.632719,9.255328



════════ BOX-PLOT – delay đến toàn bộ chuyến đã bay, theo năm ════════


,year,n,min,q1,median,q3,whisker_high,p90,p95,p99,max,mean,n_out_high,out_high_rate
0,2022,6532012,-100,-14,-5,10,46,42,79,199,3423,7.035647,592130,0.090650
1,2023,6743403,-119,-15,-6,9,45,41,78,205,4405,6.627152,607450,0.090081
2,2024,6965247,-126,-15,-6,9,45,43,83,215,3803,7.098219,665279,0.095514
3,2025,6879484,-128,-15,-6,11,50,48,91,225,4336,8.504504,659913,0.095925
4,2026,1611874,-90,-17,-7,10,50,47,90,233,3339,7.262870,150340,0.093270



════════ BOX-PLOT – delay đến của chuyến TRỄ, theo hãng và năm ════════


,year,carrier,n,min,q1,median,q3,whisker_high,p90,p95,p99,max,mean,n_out_high,out_high_rate
0,2022,9E,36473,15,25,44,89,185,161,235,724,1368,79.397582,2798,0.076714
1,2022,AA,190872,15,24,42,82,169,147,214,672,3423,75.919239,14686,0.076942
2,2022,AS,45003,15,21,33,59,116,108,151,263,908,51.271160,3920,0.087105
3,2022,B6,85373,15,27,50,95,197,164,222,382,2029,77.248041,5694,0.066696
4,2022,DL,140056,15,23,38,74,150,141,211,568,1285,69.284329,12578,0.089807
5,2022,F9,47981,15,26,45,83,168,146,201,448,1378,72.177279,3574,0.074488
6,2022,G4,38373,15,25,42,78,157,141,194,376,1919,68.276835,3106,0.080942
7,2022,HA,17131,15,20,28,45,82,82,123,303,1805,45.554433,1686,0.098418
8,2022,MQ,41891,15,23,37,72,145,127,173,326,1550,60.844262,3116,0.074384
9,2022,NK,55480,15,24,42,79,161,139,192,358,1598,66.855317,4074,0.073432


### 9. Kỹ thuật đặc biệt và bằng chứng Spark (chạy trên 1 tháng)

In [9]:
# ═══════════ KỸ THUẬT ĐẶC BIỆT (chạy trên 1 tháng nhỏ): UDF, custom aggregation, lan truyền trễ, bằng chứng Spark ═══════════
df_demo = prepare(load_months(MONTHS[:1], verbose=False)).cache()

# ── UDF có lý do nghiệp vụ: phân tầng mức độ nghiêm trọng, đối chiếu với biểu thức Spark tham chiếu ──
def severity_expr():
    a = F.col("arr_delay")
    return (F.when(F.col("cancelled") == 1, "1_cancelled").when(F.col("diverted") == 1, "2_diverted").when(a.isNull(), "0_unknown")
             .when(a < 0, "3_early").when(a < 15, "4_ontime").when(a < 30, "5_minor").when(a < 60, "6_moderate")
             .when(a < 180, "7_major").otherwise("8_severe"))
def _severity_py(a, cancelled, diverted):
    if cancelled == 1: return "1_cancelled"
    if diverted == 1: return "2_diverted"
    if a is None: return "0_unknown"
    return "3_early" if a < 0 else "4_ontime" if a < 15 else "5_minor" if a < 30 else "6_moderate" if a < 60 else "7_major" if a < 180 else "8_severe"
severity_udf = F.udf(_severity_py, T.StringType())
t = df_demo.select(severity_expr().alias("ref"), severity_udf("arr_delay", "cancelled", "diverted").alias("udf"))
section("UDF vs biểu thức tham chiếu → số dòng KHÔNG khớp / tổng"); print(t.where(F.col("ref") != F.col("udf")).count(), "/", t.count())

# ── Custom aggregation: trung bình cắt 1% hai đầu (pandas grouped-aggregate UDF). Không trộn được với hàm gộp thường → ghép bằng join ──
section("Custom aggregation: trimmed mean vs mean vs median")
try:
    import pandas as pd
    @F.pandas_udf("double")
    def trimmed_mean(v: pd.Series) -> float:
        v = v.dropna()
        if len(v) == 0: return float("nan")
        lo, hi = v.quantile(0.01), v.quantile(0.99)
        return float(v[(v >= lo) & (v <= hi)].mean())
    plain = df_demo.groupBy("year", "carrier").agg(F.count("arr_delay_ok").alias("n"), F.avg("arr_delay_ok").alias("mean"),
                                                   F.expr("percentile_approx(arr_delay_ok, 0.5, 10000)").alias("median"))
    tm = df_demo.groupBy("year", "carrier").agg(trimmed_mean("arr_delay_ok").alias("trimmed_mean"))
    peek(plain.join(tm, ["year", "carrier"]).orderBy("year", "carrier"), 20)
except Exception as e:
    print("⚠ Không chạy được pandas UDF ở đây:", type(e).__name__, "(cần pandas + pyarrow, Java 8/11/17)")

# ── Lan truyền trễ theo máy bay (cần TAIL_NUM): chuyến trước đến trễ → chuyến sau đi trễ? Chỉ xét chuyến liền kề (đến X rồi đi từ X) ──
def delay_propagation(df, min_n=100):
    if df.where(F.col("tail").isNotNull()).limit(1).count() == 0:
        print("ℹ Không có TAIL_NUM → bỏ qua"); return None
    ts = F.to_timestamp(F.concat_ws(" ", F.col("fl_date").cast("string"), F.format_string("%02d:%02d", "dep_hour", "dep_min")))
    w = Window.partitionBy("tail").orderBy("sched_ts")
    d = (df.where(F.col("tail").isNotNull() & (F.col("cancelled") == 0)).withColumn("sched_ts", ts)
           .withColumn("prev_arr_delay", F.lag("arr_delay_ok").over(w)).withColumn("prev_dest", F.lag("dest").over(w))
           .where((F.col("prev_dest") == F.col("origin")) & F.col("prev_arr_delay").isNotNull() & F.col("dep_delay_ok").isNotNull())
           .withColumn("prev_state", F.when(F.col("prev_arr_delay") < 0, "1_prev_early").when(F.col("prev_arr_delay") < 15, "2_prev_ontime")
                                      .when(F.col("prev_arr_delay") < 45, "3_prev_late15_44").otherwise("4_prev_late45plus")))
    return (d.groupBy("prev_state").agg(F.count(F.lit(1)).alias("n"), F.avg("dep_delay_ok").alias("avg_dep_delay"),
                                         F.avg((F.col("dep_delay_ok") >= 15).cast("double")).alias("dep_late15_rate"))
             .where(F.col("n") >= min_n).orderBy("prev_state"))
section("Lan truyền trễ theo máy bay"); peek(delay_propagation(df_demo))

# ── Bằng chứng kỹ thuật: partition pruning, broadcast vs sort-merge join, cache, bucketing, tối ưu truy vấn ──
EVIDENCE = {}
def plan_text(df, mode="formatted"):
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf): df.explain(mode=mode)
    return buf.getvalue()

@contextlib.contextmanager
def conf(settings):
    old = {k: spark.conf.get(k, None) for k in settings}
    for k, v in settings.items(): spark.conf.set(k, v)
    try: yield
    finally:
        for k, v in old.items(): spark.conf.unset(k) if v is None else spark.conf.set(k, v)

def save_evidence(name, text):
    os.makedirs(EVIDENCE_DIR, exist_ok=True)
    open(os.path.join(EVIDENCE_DIR, name + ".txt"), "w", encoding="utf-8").write(text)

def evidence_pruning():
    """Lọc theo year/month → số phân vùng quét giảm; plan có PartitionFilters."""
    full = spark.read.parquet(PROCESSED).select("year")
    one = spark.read.parquet(PROCESSED).where((F.col("year") == y0) & (F.col("month") == m0)).select("year")
    plan = plan_text(one.select(F.count(F.lit(1))))
    res = {"scan_partitions_no_filter": full.rdd.getNumPartitions(), "scan_partitions_with_filter": one.rdd.getNumPartitions(),
           "partition_filters": re.findall(r"PartitionFilters: \[.*?\]", plan)}
    save_evidence("pruning", json.dumps(res, indent=2) + "\n\n" + plan); return res

def evidence_joins():
    """Broadcast join vs sort-merge join cho cùng một phép ghép (kết quả phải giống nhau)."""
    out = {}
    with conf({"spark.sql.adaptive.enabled": "false"}):          # tắt AQE để plan hiển thị đúng chiến lược đã chọn
        for hint in ("broadcast", "merge"):
            df = route_excess_delay(hint=hint); plan = plan_text(df); cnt, sec = timed(lambda: df.count())
            out[hint] = {"rows": cnt, "seconds": sec, "strategy": "BroadcastHashJoin" if "BroadcastHashJoin" in plan else "SortMergeJoin" if "SortMergeJoin" in plan else "other"}
            save_evidence(f"join_{hint}", plan)
    out["same_rowcount"] = out["broadcast"]["rows"] == out["merge"]["rows"]; return out

def evidence_cache(df):
    def run(d):
        d.groupBy("carrier").agg(F.avg("arr_delay_ok")).collect(); d.groupBy("origin").agg(F.sum("operated")).collect()
        d.groupBy("dep_hour").agg(F.count(F.lit(1))).collect()
    raw = prepare(load_months(MONTHS[:1], verbose=False))
    _, t_no = timed(lambda: [run(raw) for _ in range(3)])
    c = raw.cache(); _, t_fill = timed(lambda: c.count()); _, t_c = timed(lambda: [run(c) for _ in range(3)]); c.unpersist()
    return {"no_cache_s": t_no, "cache_fill_s": t_fill, "cached_3_runs_s": t_c, "speedup_x": round(t_no / t_c, 2) if t_c else None}

def evidence_bucketing(df, buckets=8, table="aero_bts_bucketed"):
    """Bucketing theo hãng: gom nhóm theo cột bucket không cần shuffle (Exchange). Cần catalog (saveAsTable) → chỉ sống trong session."""
    cols = ["carrier", "origin", "arr_delay_ok", "operated"]
    spark.sql(f"DROP TABLE IF EXISTS {table}")
    df.select(*cols).write.mode("overwrite").bucketBy(buckets, "carrier").sortBy("carrier").saveAsTable(table)
    q = lambda d: d.groupBy("carrier").agg(F.avg("arr_delay_ok").alias("avg_delay"), F.sum("operated").alias("operated"))
    plain, buck = q(df.select(*cols)), q(spark.table(table))
    with conf({"spark.sql.adaptive.enabled": "false"}): p1, p2 = plan_text(plain), plan_text(buck)
    _, t1 = timed(lambda: plain.collect()); _, t2 = timed(lambda: buck.collect())
    save_evidence("bucketing_plain", p1); save_evidence("bucketing_bucketed", p2); spark.sql(f"DROP TABLE IF EXISTS {table}")
    return {"exchange_plain": p1.count("Exchange"), "exchange_bucketed": p2.count("Exchange"), "plain_s": t1, "bucketed_s": t2}

def evidence_optimization():
    """Bộ lọc bằng biểu thức Spark (đẩy xuống Parquet: PushedFilters) vs bằng Python UDF (không đẩy xuống được)."""
    raw = load_months(MONTHS[:1], verbose=False); a = COLMAP.get("arr_delay") or "ARR_DELAY"
    is_late = F.udf(lambda x: x is not None and x >= 15, T.BooleanType()); res = {}
    for name, d in (("native_filter", raw.where(F.col(a) >= 15).select(F.count(F.lit(1)))), ("udf_filter", raw.where(is_late(F.col(a))).select(F.count(F.lit(1))))):
        plan = plan_text(d); rows, sec = timed(lambda: d.collect()[0][0]); save_evidence(f"optimization_{name}", plan)
        res[name] = {"late_flights": rows, "seconds": sec, "pushed_filters": re.findall(r"PushedFilters: \[.*?\]", plan)}
    res["same_result"] = res["native_filter"]["late_flights"] == res["udf_filter"]["late_flights"]; return res

section("BẰNG CHỨNG KỸ THUẬT SPARK")
for fn, args in ((evidence_pruning, ()), (evidence_joins, ()), (evidence_cache, (df_demo,)), (evidence_bucketing, (df_demo,)), (evidence_optimization, ())):
    try:
        EVIDENCE[fn.__name__] = fn(*args); print(fn.__name__, "→", json.dumps(EVIDENCE[fn.__name__], ensure_ascii=False, default=str))
    except Exception as e:
        print(fn.__name__, "✗", type(e).__name__, str(e)[:300])
save_evidence("summary", json.dumps(EVIDENCE, indent=2, ensure_ascii=False, default=str))
print("\nPlan chi tiết đã lưu trong:", os.path.abspath(EVIDENCE_DIR))


26/10/28 08:43:45 WARN DataSource: [COLUMN_ALREADY_EXISTS] The column `month` already exists. Consider to choose another name or rename the existing column.



════════ UDF vs biểu thức tham chiếu → số dòng KHÔNG khớp / tổng ════════


0 / 537902

════════ Custom aggregation: trimmed mean vs mean vs median ════════


,year,carrier,n,mean,median,trimmed_mean
0,2022,9E,20259,1.984254,-10.0,-1.704643
1,2022,AA,66370,-1.075832,-11.0,-4.810863
2,2022,AS,15204,5.233294,-4.0,3.330340
3,2022,B6,19192,18.721082,-2.0,15.010150
4,2022,DL,66573,-0.228486,-9.0,-3.111283
5,2022,F9,11486,10.955163,-4.0,7.998491
6,2022,G4,7975,12.243135,-2.0,9.870683
7,2022,HA,5640,6.884220,-2.0,4.799349
8,2022,MQ,21497,-1.326464,-10.0,-3.427143
9,2022,NK,16933,8.520168,-5.0,5.690738



════════ Lan truyền trễ theo máy bay ════════


...

26/10/28 08:44:13 WARN DataSource: [COLUMN_ALREADY_EXISTS] The column `month` already exists. Consider to choose another name or rename the existing column.
26/10/28 08:44:16 WARN CacheManager: Asked to cache already cached data.


evidence_cache → {"no_cache_s": 2.21, "cache_fill_s": 0.12, "cached_3_runs_s": 1.38, "speedup_x": 1.6}


evidence_bucketing → {"exchange_plain": 2, "exchange_bucketed": 0, "plain_s": 2.21, "bucketed_s": 0.52}


26/10/28 08:44:30 WARN DataSource: [COLUMN_ALREADY_EXISTS] The column `month` already exists. Consider to choose another name or rename the existing column.


evidence_optimization → {"native_filter": {"late_flights": 97052, "seconds": 1.77, "pushed_filters": ["PushedFilters: [IsNotNull(ARR_DELAY), GreaterThanOrEqual(ARR_DELAY,15.0)]"]}, "udf_filter": {"late_flights": 97052, "seconds": 1.71, "pushed_filters": []}, "same_result": true}

Plan chi tiết đã lưu trong: /opt/spark_jobs/evidence
